# Experiment Visualization for Report Writing

This notebook is designed around the current direction of the project: **showing symmetry-based parameter sharing as an inductive bias** rather than as a mere compression trick.

The notebook focuses on three report-friendly questions.

1. Does symmetry-aware parameter sharing improve **sample efficiency**?
2. Does it reduce **symmetry error** and produce a cleaner hypothesis space?
3. Does it remain competitive or advantageous under **parameter constraints**?

To support those claims, the notebook creates:

- DeepSet summary tables, heatmaps, and parameter-efficiency plots
- Graph experiment aggregations from saved logs and validation-loss heatmaps
- 3Body quantitative comparisons and qualitative rollout visualizations

It also writes intermediate summary tables into `figures/summary_tables` so they can be reused in the report.

In [ ]:
from pathlib import Path
import re
import json
import math
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

ROOT = Path.cwd()
assert ROOT.name == 'Optimizing-Hypothesis-Space-via-Group-Representation-Intertwiner', (
    'Please run this notebook from the Optimizing-Hypothesis-Space-via-Group-Representation-Intertwiner directory.'
)

FIG_DIR = ROOT / 'figures'
TABLE_DIR = FIG_DIR / 'summary_tables'
FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def save_table(df, name):
    csv_path = TABLE_DIR / f'{name}.csv'
    df.to_csv(csv_path, index=False)
    print(f'Saved table -> {csv_path}')

def save_figure(name):
    path = FIG_DIR / name
    plt.savefig(path, bbox_inches='tight', dpi=220)
    print(f'Saved figure -> {path}')

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

shared_palette = {
    'shared': '#1f77b4',
    'vanilla': '#d62728'
}

task_titles = {
    'DeepSet': 'Task 1: Permutation-Invariant Set Function',
    'Graph': 'Task 2: Graph Classification',
    '3Body': 'Task 3: Symmetric Physical System Prediction'
}

In [ ]:
# ------------------------------------------------------------------
# Task 1: DeepSet aggregation and visualization
# ------------------------------------------------------------------
deep_result = load_pickle(ROOT / 'DeepSetProblem' / 'total_result.pkl')

deep_rows = []
for n_block, by_size in deep_result.items():
    for data_size, metrics in by_size.items():
        deep_rows.append({
            'task': 'DeepSet',
            'n_block': int(n_block),
            'data_size': int(data_size),
            'vanilla_loss': float(metrics['vanilla_loss']),
            'shared_loss': float(metrics['shared_loss']),
            'vanilla_symmetry': float(metrics['vanilla_symmetry']),
            'shared_symmetry': float(metrics['shared_symmetry']),
            'vanilla_size': int(metrics['vanilla_size']),
            'shared_size': int(metrics['shared_size']),
        })

deep_df = pd.DataFrame(deep_rows).sort_values(['n_block', 'data_size']).reset_index(drop=True)
deep_df['loss_gain'] = deep_df['vanilla_loss'] - deep_df['shared_loss']
deep_df['symmetry_gain'] = deep_df['vanilla_symmetry'] - deep_df['shared_symmetry']
deep_df['size_ratio'] = deep_df['vanilla_size'] / deep_df['shared_size']
save_table(deep_df, 'deepset_summary')
deep_df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for n_block in sorted(deep_df['n_block'].unique()):
    sub = deep_df[deep_df['n_block'] == n_block]
    axes[0].plot(sub['data_size'], sub['vanilla_loss'], marker='o', linestyle='--', alpha=0.7, label=f'vanilla, n={n_block}')
    axes[0].plot(sub['data_size'], sub['shared_loss'], marker='o', linewidth=2.5, label=f'shared, n={n_block}')
    axes[1].plot(sub['data_size'], sub['vanilla_symmetry'], marker='o', linestyle='--', alpha=0.7, label=f'vanilla, n={n_block}')
    axes[1].plot(sub['data_size'], sub['shared_symmetry'], marker='o', linewidth=2.5, label=f'shared, n={n_block}')

axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_title('DeepSet: Test Loss vs Data Size')
axes[0].set_xlabel('Training data size')
axes[0].set_ylabel('Test loss')

axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title('DeepSet: Symmetry Error vs Data Size')
axes[1].set_xlabel('Training data size')
axes[1].set_ylabel('Symmetry error')

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout()
save_figure('deepset_curves.png')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
loss_heat = deep_df.pivot(index='n_block', columns='data_size', values='loss_gain')
sym_heat = deep_df.pivot(index='n_block', columns='data_size', values='symmetry_gain')
sns.heatmap(loss_heat, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=axes[0])
axes[0].set_title('DeepSet: (Vanilla loss - Shared loss)')
axes[0].set_xlabel('Training data size')
axes[0].set_ylabel('Number of blocks')
sns.heatmap(sym_heat, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=axes[1])
axes[1].set_title('DeepSet: (Vanilla symmetry error - Shared symmetry error)')
axes[1].set_xlabel('Training data size')
axes[1].set_ylabel('Number of blocks')
plt.tight_layout()
save_figure('deepset_heatmaps.png')
plt.show()

scatter_rows = []
for _, row in deep_df.iterrows():
    scatter_rows.append({'model': 'vanilla', 'n_block': row['n_block'], 'data_size': row['data_size'], 'params': row['vanilla_size'], 'test_loss': row['vanilla_loss']})
    scatter_rows.append({'model': 'shared', 'n_block': row['n_block'], 'data_size': row['data_size'], 'params': row['shared_size'], 'test_loss': row['shared_loss']})
scatter_df = pd.DataFrame(scatter_rows)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=scatter_df,
    x='params',
    y='test_loss',
    hue='model',
    size='n_block',
    sizes=(60, 260),
    palette=shared_palette,
)
plt.xscale('log')
plt.yscale('log')
plt.title('DeepSet: Parameter Efficiency Frontier')
plt.xlabel('Trainable parameters')
plt.ylabel('Test loss')
plt.tight_layout()
save_figure('deepset_parameter_efficiency.png')
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Task 2: Graph log aggregation and visualization
# ------------------------------------------------------------------
graph_log_dir = ROOT / 'GraphProblem' / 'logs'
graph_rows = []
curve_store = {}

pattern = re.compile(r'(?P<model>shared|vanilla)_log_(?P<width>\d+)_(?P<data>\d+)\.pkl')

for path in sorted(graph_log_dir.glob('*.pkl')):
    match = pattern.match(path.name)
    if not match:
        continue
    obj = load_pickle(path)
    model = match.group('model')
    width = int(match.group('width'))
    data_size = int(match.group('data'))
    train_loss = obj.get('train_loss', [])
    valid_loss = obj.get('valid_loss', [])
    final_train = float(train_loss[-1]) if train_loss else np.nan
    final_valid = float(valid_loss[-1]) if valid_loss else np.nan
    best_valid = float(np.min(valid_loss)) if valid_loss else np.nan
    best_epoch = int(np.argmin(valid_loss) + 1) if valid_loss else np.nan

    graph_rows.append({
        'task': 'Graph',
        'model': model,
        'width_tag': width,
        'data_size': data_size,
        'epochs_ran': len(train_loss),
        'final_train_loss': final_train,
        'final_valid_loss': final_valid,
        'best_valid_loss': best_valid,
        'best_epoch': best_epoch,
    })
    curve_store[(model, width, data_size)] = {'train_loss': train_loss, 'valid_loss': valid_loss}

graph_df = pd.DataFrame(graph_rows).sort_values(['width_tag', 'data_size', 'model']).reset_index(drop=True)
graph_pivot = graph_df.pivot_table(index=['width_tag', 'data_size'], columns='model', values='best_valid_loss').reset_index()
graph_pivot['valid_gain'] = graph_pivot['vanilla'] - graph_pivot['shared']
save_table(graph_df, 'graph_log_summary')
save_table(graph_pivot, 'graph_valid_gain_summary')
graph_pivot

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=False)
axes = axes.flatten()

panel_configs = [(2, 500), (2, 4110), (3, 500), (3, 4110), (5, 500), (7, 4110)]
for ax, (width, data_size) in zip(axes, panel_configs):
    for model in ['vanilla', 'shared']:
        curves = curve_store.get((model, width, data_size))
        if curves is None:
            continue
        ax.plot(curves['train_loss'], label=f'{model} train', color=shared_palette[model], alpha=0.9)
        if curves['valid_loss']:
            ax.plot(curves['valid_loss'], label=f'{model} valid', color=shared_palette[model], linestyle='--', alpha=0.7)
    ax.set_title(f'width={width}, data={data_size}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.suptitle('Graph: Training Dynamics Across Representative Conditions', y=1.02)
plt.tight_layout()
save_figure('graph_training_dynamics.png')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
shared_heat = graph_df[graph_df['model'] == 'shared'].pivot(index='width_tag', columns='data_size', values='best_valid_loss')
vanilla_heat = graph_df[graph_df['model'] == 'vanilla'].pivot(index='width_tag', columns='data_size', values='best_valid_loss')
gain_heat = graph_pivot.pivot(index='width_tag', columns='data_size', values='valid_gain')
sns.heatmap(gain_heat, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=axes[0])
axes[0].set_title('Graph: (Vanilla best val loss - Shared best val loss)')
axes[0].set_xlabel('Training data size')
axes[0].set_ylabel('Model width tag')
epoch_heat = graph_df[graph_df['model'] == 'shared'].pivot(index='width_tag', columns='data_size', values='best_epoch')
sns.heatmap(epoch_heat, annot=True, fmt='.0f', cmap='Blues', ax=axes[1])
axes[1].set_title('Graph: Shared model best epoch')
axes[1].set_xlabel('Training data size')
axes[1].set_ylabel('Model width tag')
plt.tight_layout()
save_figure('graph_heatmaps.png')
plt.show()

graph_note = pd.DataFrame([
    {
        'note': 'Graph artifacts currently store training logs but not fixed test predictions or dataset splits.',
        'implication': 'For the report, use these figures mainly for training stability and validation-loss trend claims.'
    }
])
graph_note

In [ ]:
# ------------------------------------------------------------------
# Task 3: 3Body aggregation, evaluation, and rollout visualization
# ------------------------------------------------------------------
class EquivariantLinear2D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        scale = 1.0 / np.sqrt(in_channels * 3)
        self.A = nn.Parameter(torch.randn(out_channels, in_channels) * scale)
        self.B = nn.Parameter(torch.randn(out_channels, in_channels) * scale)
        self.C = nn.Parameter(torch.randn(out_channels, in_channels) * scale)
        self.D = nn.Parameter(torch.randn(out_channels, in_channels) * scale)
        self.E = nn.Parameter(torch.randn(out_channels, in_channels) * scale)
        self.bias_sym = nn.Parameter(torch.zeros(out_channels))
        self.bias_third = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x):
        row1 = torch.cat([self.A, self.B, self.C], dim=1)
        row2 = torch.cat([self.B, self.A, self.C], dim=1)
        row3 = torch.cat([self.D, self.D, self.E], dim=1)
        W = torch.cat([row1, row2, row3], dim=0)
        b = torch.cat([self.bias_sym, self.bias_sym, self.bias_third], dim=0)
        return x @ W.t() + b

def build_vanilla_from_state_dict(state_dict):
    linear_keys = sorted({int(k.split('.')[0]) for k in state_dict if k.endswith('weight')})
    layers = []
    for idx, key in enumerate(linear_keys):
        weight = state_dict[f'{key}.weight']
        out_dim, in_dim = weight.shape
        layers.append(nn.Linear(in_dim, out_dim))
        if idx < len(linear_keys) - 1:
            layers.append(nn.ReLU())
    model = nn.Sequential(*layers)
    model.load_state_dict(state_dict)
    return model

def build_equivariant_from_state_dict(state_dict):
    layer_keys = sorted({int(k.split('.')[0]) for k in state_dict if k.endswith('.A')})
    layers = []
    for idx, key in enumerate(layer_keys):
        out_dim, in_dim = state_dict[f'{key}.A'].shape
        layers.append(EquivariantLinear2D(in_dim, out_dim))
        if idx < len(layer_keys) - 1:
            layers.append(nn.ReLU())
    model = nn.Sequential(*layers)
    model.load_state_dict(state_dict)
    return model

def architecture_signature(state_dict):
    parts = []
    if any(k.endswith('.A') for k in state_dict):
        layer_keys = sorted({int(k.split('.')[0]) for k in state_dict if k.endswith('.A')})
        for key in layer_keys:
            out_dim, in_dim = state_dict[f'{key}.A'].shape
            parts.append(f'E({in_dim}->{out_dim})')
    else:
        layer_keys = sorted({int(k.split('.')[0]) for k in state_dict if k.endswith('weight')})
        for key in layer_keys:
            out_dim, in_dim = state_dict[f'{key}.weight'].shape
            parts.append(f'L({in_dim}->{out_dim})')
    return ' - '.join(parts)

def swap_first_two_particles(x):
    x = x.clone()
    x[..., 0:4], x[..., 4:8] = x[..., 4:8].clone(), x[..., 0:4].clone()
    return x

def symmetry_error_3body(model, X, batch_size=512):
    model.eval()
    values = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            batch = X[start:start+batch_size]
            out1 = model(batch)
            out2 = model(swap_first_two_particles(batch))
            out2_swapped = swap_first_two_particles(out2)
            err = ((out1 - out2_swapped) ** 2).mean(dim=-1)
            values.append(err.cpu())
    return torch.cat(values).mean().item()

X_test_3body = torch.load(ROOT / '3BodyProblem' / 'data' / 'X_test.pt', map_location=device)
Y_test_3body = torch.load(ROOT / '3BodyProblem' / 'data' / 'Y_test.pt', map_location=device)

three_body_models = {}
three_body_rows = []
for model_path in sorted((ROOT / '3BodyProblem' / 'models').glob('*_0408.pt')):
    name = model_path.stem
    state_dict = torch.load(model_path, map_location=device)
    if 'equiv' in name:
        model = build_equivariant_from_state_dict(state_dict)
    else:
        model = build_vanilla_from_state_dict(state_dict)
    three_body_models[name] = model
    model.to(device)
    model.eval()
    with torch.no_grad():
        pred = model(X_test_3body)
        test_loss = nn.MSELoss()(pred, Y_test_3body).item()
    sym_err = symmetry_error_3body(model, X_test_3body)
    family = 'shared' if 'equiv' in name else 'vanilla'
    variant = 'checkpoint_1' if 'model1' in name else 'checkpoint_2'
    three_body_rows.append({
        'task': '3Body',
        'name': name,
        'model_family': family,
        'variant': variant,
        'test_loss': test_loss,
        'symmetry_error': sym_err,
        'params': count_parameters(model),
        'architecture': architecture_signature(state_dict),
    })

three_body_df = pd.DataFrame(three_body_rows).sort_values(['variant', 'model_family']).reset_index(drop=True)
save_table(three_body_df, 'three_body_summary')
three_body_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.barplot(data=three_body_df, x='variant', y='test_loss', hue='model_family', palette=shared_palette, ax=axes[0])
axes[0].set_title('3Body: Test Loss Comparison')
axes[0].set_xlabel('Checkpoint label')
axes[0].set_ylabel('Test loss')

sns.barplot(data=three_body_df, x='variant', y='symmetry_error', hue='model_family', palette=shared_palette, ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('3Body: Symmetry Error Comparison')
axes[1].set_xlabel('Checkpoint label')
axes[1].set_ylabel('Symmetry error')

sns.scatterplot(data=three_body_df, x='params', y='test_loss', hue='model_family', style='variant', s=180, palette=shared_palette, ax=axes[2])
axes[2].set_xscale('log')
axes[2].set_title('3Body: Parameter Efficiency')
axes[2].set_xlabel('Trainable parameters')
axes[2].set_ylabel('Test loss')

for ax in axes[1:]:
    if ax.legend_ is not None:
        ax.legend_.remove()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)
plt.tight_layout()
save_figure('3body_summary.png')
plt.show()

def find_contiguous_segment(X, Y, min_length=20, atol=1e-6):
    current_start = 0
    current_length = 1
    best = (0, 1)
    for i in range(len(X) - 1):
        if torch.allclose(Y[i], X[i+1], atol=atol, rtol=0):
            current_length += 1
        else:
            if current_length > best[1]:
                best = (current_start, current_length)
            current_start = i + 1
            current_length = 1
    if current_length > best[1]:
        best = (current_start, current_length)
    if best[1] < min_length:
        return None
    return best

segment = find_contiguous_segment(X_test_3body, Y_test_3body, min_length=15)
print('contiguous segment:', segment)

if segment is not None:
    start, length = segment
    rollout_len = min(length, 40)
    gt_states = [X_test_3body[start].detach().cpu().numpy()]
    for idx in range(start, start + rollout_len):
        gt_states.append(Y_test_3body[idx].detach().cpu().numpy())
    gt_states = np.stack(gt_states)

    vanilla_candidates = [k for k in three_body_models if 'vanilla' in k]
    equiv_candidates = [k for k in three_body_models if 'equiv' in k]
    vanilla_key = sorted(vanilla_candidates)[-1]
    equiv_key = sorted(equiv_candidates)[-1]
    selected_models = {
        'Ground Truth': None,
        f'Vanilla ({vanilla_key})': three_body_models[vanilla_key],
        f'Equivariant ({equiv_key})': three_body_models[equiv_key],
    }

    predictions = {}
    init_state = X_test_3body[start:start+1]
    for label, model in selected_models.items():
        if model is None:
            predictions[label] = gt_states
            continue
        model.eval()
        curr = init_state.clone()
        states = [curr[0].detach().cpu().numpy()]
        with torch.no_grad():
            for _ in range(rollout_len):
                curr = model(curr)
                states.append(curr[0].detach().cpu().numpy())
        predictions[label] = np.stack(states)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    particle_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    for ax, (label, states) in zip(axes, predictions.items()):
        states = states.reshape(len(states), 3, 4)
        for particle_idx, color in enumerate(particle_colors):
            xy = states[:, particle_idx, :2]
            ax.plot(xy[:, 0], xy[:, 1], color=color, linewidth=2, label=f'particle {particle_idx+1}')
            ax.scatter(xy[0, 0], xy[0, 1], color=color, s=50)
        ax.set_title(label)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_aspect('equal', adjustable='box')
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)
    fig.suptitle('3Body: Qualitative Rollout Comparison', y=1.02)
    plt.tight_layout()
    save_figure('3body_rollout.png')
    plt.show()
else:
    print('Could not find a long contiguous segment in X_test/Y_test for rollout visualization.')

## Recommended Figure Set for the Report

If you want a compact main-text figure list, the highest-signal candidates are:

- `deepset_curves.png`: strongest evidence for sample efficiency and symmetry preservation
- `deepset_parameter_efficiency.png`: supports the inductive-bias vs compression discussion
- `graph_heatmaps.png`: compact summary for the graph experiment trends
- `3body_summary.png`: direct quantitative comparison for the physical-system task
- `3body_rollout.png`: qualitative figure for intuitive physical consistency

The CSV files in `figures/summary_tables` can be reused to build tables in the final report.